# CCTV shop analytics: which dashboard numbers survive contact with the video

A viral demo runs coffee-shop analytics off existing CCTV and puts per-person
dwell time, movement patterns, wait times and named per-staff tallies on a live
dashboard. Drawing boxes is not the interesting part. The interesting question is
**which of those numbers are actually recoverable from the video, and which ones
quietly depend on holding one person's identity for their entire visit.**

MOT17 ships ground-truth person tracks, so every number below is measured against
a true value instead of asserted.

Anonymous integer IDs only. No face recognition, no enrolment, no naming.


## 1. What is actually in the attached dataset

Walked with no depth cap, classified by `os.path.basename` only.

In [ ]:
# =====================================================================
# CELL 1 - walk the attached dataset. NO depth cap. Print real filenames.
# =====================================================================
import os, sys, collections

ROOT = "/kaggle/input"
print("python", sys.version.split()[0])
print("listing", ROOT, "->", os.listdir(ROOT))

by_ext = collections.Counter()
by_kind = collections.Counter()
leafdirs = {}
n_files = 0
total_bytes = 0


def kind_of(path):
    """Classify by BASENAME only - never by searching the whole path."""
    base = os.path.basename(path)
    _stem, ext = os.path.splitext(base)
    ext = ext.lower()
    if ext in (".jpg", ".jpeg", ".png"):
        return "image"
    if base == "gt.txt":
        return "groundtruth"
    if base == "det.txt":
        return "detections"
    if base == "seqinfo.ini":
        return "seqinfo"
    if ext == ".txt":
        return "text-other"
    return "other" + ext


for dirpath, dirnames, filenames in os.walk(ROOT):
    dirnames.sort()
    filenames.sort()
    if filenames:
        leafdirs[dirpath] = filenames
    for fn in filenames:
        n_files += 1
        p = os.path.join(dirpath, fn)
        by_ext[os.path.splitext(fn)[1].lower()] += 1
        by_kind[kind_of(p)] += 1
        try:
            total_bytes += os.path.getsize(p)
        except OSError:
            pass

print("\ntotal files %d  total %.2f GB" % (n_files, total_bytes / 1e9))
print("by extension:", dict(by_ext))
print("by kind (basename-classified):", dict(by_kind))

print("\n==== every directory holding files (%d), with real filenames ====" % len(leafdirs))
for d in sorted(leafdirs):
    fns = leafdirs[d]
    rel = os.path.relpath(d, ROOT)
    depth = rel.count(os.sep) + 1
    if len(fns) > 8:
        show = fns[:4] + ["...(%d more)" % (len(fns) - 8)] + fns[-4:]
    else:
        show = fns
    print("  [depth %d] %s  (%d files)" % (depth, rel, len(fns)))
    print("            " + ", ".join(show))

assert n_files > 0, "the attached dataset is empty"
assert by_kind["groundtruth"] > 0, "no gt.txt found - nothing to measure against"


## 2. Configuration

Nothing here is trained. The split is **by sequence**: the dev sequences choose the detector, the test sequences carry every reported number.

In [ ]:
# =====================================================================
# CELL 2 - imports, seeds, config
# =====================================================================
import os, re, json, time, math, random, collections, configparser
import numpy as np
import torch
from scipy.optimize import linear_sum_assignment

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "device", DEV)
if DEV == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

CFG = json.loads(r"""
{
  "dev_sequences":  ["MOT17-09", "MOT17-02"],
  "test_sequences": ["MOT17-04", "MOT17-05", "MOT17-10", "MOT17-11", "MOT17-13"],
  "gt_variant": "FRCNN",
  "public_det_variant": "SDP",
  "det_min_size": 1000,
  "det_max_size": 1800,
  "det_score_keep": 0.10,
  "eval_iou_threshold": 0.5,
  "distractor_classes": [2, 7, 8, 12],
  "pedestrian_class": 1,
  "zone_x": [0.30, 0.70],
  "zone_y": [0.40, 1.00],
  "line_x_frac": 0.50,
  "bars": {"idf1_min": 0.60, "hota_min": 0.50}
}
""")
print(json.dumps(CFG, indent=1))

# PILOT_ONLY is flipped to False for the full run (see cell "RUN").
PILOT_ONLY = True
OUT = "/kaggle/working"
T_START = time.time()


## 3. Sequences

MOT17's *train* split is the only one that ships ground-truth tracks, so it is the only split on which any of these numbers can be measured rather than asserted.

In [ ]:
# =====================================================================
# CELL 3 - locate MOT17 train sequences, parse seqinfo + ground truth files
# =====================================================================
def find_mot_root(root="/kaggle/input"):
    """Find the dir D such that D/<split>/<seq>/gt/gt.txt exists."""
    hits = []
    for dirpath, dirnames, filenames in os.walk(root):
        if os.path.basename(dirpath) == "gt" and "gt.txt" in filenames:
            seqdir = os.path.dirname(dirpath)             # .../<split>/<seq>
            splitdir = os.path.dirname(seqdir)            # .../<split>
            hits.append((os.path.dirname(splitdir), os.path.basename(splitdir),
                         os.path.basename(seqdir)))
    assert hits, "found no <seq>/gt/gt.txt anywhere under " + root
    roots = collections.Counter(h[0] for h in hits)
    mot_root = roots.most_common(1)[0][0]
    print("mot_root =", mot_root)
    print("splits carrying gt:", sorted(set(h[1] for h in hits if h[0] == mot_root)))
    return mot_root, hits


MOT_ROOT, GT_HITS = find_mot_root()

SEQDIRS = {}     # "MOT17-02" -> {"FRCNN": path, "SDP": path, "DPM": path}
for rootdir, split, seqname in GT_HITS:
    if rootdir != MOT_ROOT:
        continue
    m = re.match(r"^(MOT\d+-\d+)-(\w+)$", seqname)
    assert m, "unexpected sequence folder name: " + seqname
    SEQDIRS.setdefault(m.group(1), {})[m.group(2)] = os.path.join(rootdir, split, seqname)

print("\nsequences with ground truth:", sorted(SEQDIRS))
for k in sorted(SEQDIRS):
    print("  %s -> detector variants %s" % (k, sorted(SEQDIRS[k])))


def pick_variant(seq, want):
    d = SEQDIRS[seq]
    return d[want] if want in d else d[sorted(d)[0]]


def read_seqinfo(seqdir):
    p = os.path.join(seqdir, "seqinfo.ini")
    if os.path.exists(p):
        cp = configparser.ConfigParser()
        cp.read(p)
        s = cp["Sequence"]
        return {"fps": float(s["frameRate"]), "n_frames": int(s["seqLength"]),
                "width": int(s["imWidth"]), "height": int(s["imHeight"]),
                "imext": s.get("imExt", ".jpg")}
    from PIL import Image
    imgs = sorted(os.listdir(os.path.join(seqdir, "img1")))
    w, h = Image.open(os.path.join(seqdir, "img1", imgs[0])).size
    return {"fps": 30.0, "n_frames": len(imgs), "width": w, "height": h,
            "imext": os.path.splitext(imgs[0])[1]}


def read_mot_txt(path, is_gt):
    """frame -> list of rows. GT rows are (id, box, flag, cls, vis)."""
    out = collections.defaultdict(list)
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            p = line.split(",")
            fr = int(float(p[0]))
            tid = int(float(p[1]))
            x, y, w, h = float(p[2]), float(p[3]), float(p[4]), float(p[5])
            box = (x, y, x + w, y + h)
            if is_gt:
                flag = int(float(p[6]))
                cls = int(float(p[7]))
                vis = float(p[8]) if len(p) > 8 else 1.0
                out[fr].append((tid, box, flag, cls, vis))
            else:
                score = float(p[6]) if len(p) > 6 and p[6] not in ("", "-1") else 1.0
                out[fr].append((tid, box, score))
    return dict(out)


SEQINFO = {}
for base in sorted(SEQDIRS):
    SEQINFO[base] = read_seqinfo(pick_variant(base, CFG["gt_variant"]))
    print(base, SEQINFO[base])

ALL_SEQS = CFG["dev_sequences"] + CFG["test_sequences"]
missing = [s for s in ALL_SEQS if s not in SEQDIRS]
assert not missing, "CFG names sequences absent from the dataset: %s" % missing
# the split is BY SEQUENCE: no frame of a test sequence is used for tuning
assert not (set(CFG["dev_sequences"]) & set(CFG["test_sequences"])), "dev/test overlap"
print("\ndev  (threshold choice) :", CFG["dev_sequences"])
print("test (reported numbers) :", CFG["test_sequences"])
print("total test frames       :", sum(SEQINFO[s]["n_frames"] for s in CFG["test_sequences"]))


## 4. Ground truth and the official prediction preprocessing

In [ ]:
# =====================================================================
# CELL 4 - geometry helpers, GT loader, official prediction preprocessing
# =====================================================================
def iou_matrix(a, b):
    """a: (N,4) xyxy, b: (M,4) xyxy -> (N,M) IoU."""
    a = np.asarray(a, dtype=np.float64).reshape(-1, 4)
    b = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float64)
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0])
    iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2])
    iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    iw = np.clip(ix2 - ix1, 0, None)
    ih = np.clip(iy2 - iy1, 0, None)
    inter = iw * ih
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    union = aa[:, None] + ab[None, :] - inter
    return np.where(union > 0, inter / np.maximum(union, 1e-12), 0.0)


EMPTY = {"ids": np.zeros(0, np.int64), "boxes": np.zeros((0, 4), np.float64)}


def load_gt(seq):
    """Raw GT rows plus the pedestrian-only evaluation GT.

    MOT17 gt.txt columns: frame, id, x, y, w, h, flag, class, visibility.
    Official preprocessing: evaluate against class==pedestrian with flag!=0.
    Predictions landing on a DISTRACTOR class (person-on-vehicle, static
    person, distractor, reflection) are deleted, not charged as false
    positives -- otherwise MOTA is unfairly punished for correct detections
    of people the benchmark chose not to annotate as targets.
    """
    seqdir = pick_variant(seq, CFG["gt_variant"])
    raw = read_mot_txt(os.path.join(seqdir, "gt", "gt.txt"), is_gt=True)
    nf = SEQINFO[seq]["n_frames"]
    ev, allrows = {}, {}
    for fr in range(1, nf + 1):
        rows = raw.get(fr, [])
        allrows[fr] = rows
        keep = [(t, b, vis) for (t, b, fl, c, vis) in rows
                if c == CFG["pedestrian_class"] and fl != 0]
        ev[fr] = {"ids": np.array([k[0] for k in keep], dtype=np.int64),
                  "boxes": np.array([k[1] for k in keep], dtype=np.float64).reshape(-1, 4),
                  "vis": np.array([k[2] for k in keep], dtype=np.float64)}
    n_det = sum(len(ev[f]["ids"]) for f in ev)
    n_ids = len(set(int(i) for f in ev for i in ev[f]["ids"]))
    return {"eval": ev, "raw": allrows, "n_frames": nf, "seq": seq,
            "fps": SEQINFO[seq]["fps"], "width": SEQINFO[seq]["width"],
            "height": SEQINFO[seq]["height"], "n_gt_dets": n_det, "n_gt_ids": n_ids}


def preprocess_predictions(gt, pred):
    """Delete predictions matched to distractor-class GT boxes."""
    dcls = list(CFG["distractor_classes"])
    out, removed = {}, 0
    for fr in range(1, gt["n_frames"] + 1):
        pf = pred.get(fr, EMPTY)
        ids = np.asarray(pf["ids"], np.int64)
        boxes = np.asarray(pf["boxes"], np.float64).reshape(-1, 4)
        rows = gt["raw"].get(fr, [])
        if len(ids) and rows:
            gboxes = np.array([r[1] for r in rows], dtype=np.float64)
            gcls = np.array([r[3] for r in rows], dtype=np.int64)
            sim = iou_matrix(gboxes, boxes)
            sim[sim < CFG["eval_iou_threshold"] - 1e-12] = 0.0
            r, c = linear_sum_assignment(-sim)
            ok = sim[r, c] > 1e-12
            r, c = r[ok], c[ok]
            drop = c[np.isin(gcls[r], dcls)]
            if len(drop):
                mask = np.ones(len(ids), dtype=bool)
                mask[drop] = False
                ids, boxes = ids[mask], boxes[mask]
                removed += int(len(drop))
        out[fr] = {"ids": ids, "boxes": boxes}
    return out, removed


print("cell 4 ok")


## 5. Tracking metrics: HOTA, IDF1, MOTA, ID switches

Follows TrackEval's definitions, implemented here so there is no dependency to drift.

In [ ]:
# =====================================================================
# CELL 5 - tracking metrics: HOTA, IDF1, CLEAR/MOTA, ID switches
# Implemented to follow TrackEval's definitions (no TrackEval dependency).
# =====================================================================
EPS = np.finfo("float").eps
ALPHAS = np.arange(0.05, 0.99, 0.05)      # 19 localisation thresholds, as HOTA


def _index_ids(gt_f, pr_f, frames):
    """Dense 0..n-1 indices for gt and predicted identities."""
    g = sorted(set(int(i) for f in frames for i in gt_f[f]["ids"]))
    p = sorted(set(int(i) for f in frames for i in pr_f[f]["ids"]))
    return {v: k for k, v in enumerate(g)}, {v: k for k, v in enumerate(p)}, g, p


def _per_frame(gt_f, pr_f, frames, gmap, pmap):
    """Yield (frame, gt_idx, pred_idx, similarity_matrix)."""
    for f in frames:
        gi = np.array([gmap[int(i)] for i in gt_f[f]["ids"]], dtype=np.int64)
        pi = np.array([pmap[int(i)] for i in pr_f[f]["ids"]], dtype=np.int64)
        sim = iou_matrix(gt_f[f]["boxes"], pr_f[f]["boxes"])
        yield f, gi, pi, sim


def eval_sequence(gt_f, pr_f, n_frames):
    """All tracking metrics for one sequence.

    gt_f / pr_f: frame -> {"ids": (N,), "boxes": (N,4)}
    Returns a dict of scalars plus per-GT-identity diagnostics.
    """
    frames = list(range(1, n_frames + 1))
    gmap, pmap, glist, plist = _index_ids(gt_f, pr_f, frames)
    ng, npr = len(gmap), len(pmap)

    gt_count = np.zeros(max(ng, 1))
    pr_count = np.zeros(max(npr, 1))
    potential = np.zeros((max(ng, 1), max(npr, 1)))       # IoU-weighted co-occurrence
    cooc_at_half = np.zeros((max(ng, 1), max(npr, 1)))    # count with IoU >= 0.5

    cache = []
    for f, gi, pi, sim in _per_frame(gt_f, pr_f, frames, gmap, pmap):
        cache.append((gi, pi, sim))
        if len(gi):
            np.add.at(gt_count, gi, 1)
        if len(pi):
            np.add.at(pr_count, pi, 1)
        if len(gi) and len(pi):
            np.add.at(potential, (gi[:, None], pi[None, :]), sim)
            np.add.at(cooc_at_half, (gi[:, None], pi[None, :]),
                      (sim >= CFG["eval_iou_threshold"] - EPS).astype(np.float64))

    n_gt_dets = float(gt_count.sum()) if ng else 0.0
    n_pr_dets = float(pr_count.sum()) if npr else 0.0

    # ---------------------------------------------------------------- HOTA
    if ng == 0 or npr == 0:
        hota = det_a = ass_a = 0.0
        hota_at_half = 0.0
    else:
        denom = gt_count[:, None] + pr_count[None, :] - potential
        global_align = potential / np.maximum(denom, EPS)
        per_alpha, per_alpha_det, per_alpha_ass = [], [], []
        for alpha in ALPHAS:
            matches = np.zeros((ng, npr))
            tp = fn = fp = 0.0
            for gi, pi, sim in cache:
                if len(gi) and len(pi):
                    score = global_align[gi[:, None], pi[None, :]] * sim
                    r, c = linear_sum_assignment(-score)
                    ok = sim[r, c] > alpha - EPS
                    r, c = r[ok], c[ok]
                    np.add.at(matches, (gi[r], pi[c]), 1)
                    tp += len(r)
                    fn += len(gi) - len(r)
                    fp += len(pi) - len(c)
                else:
                    fn += len(gi)
                    fp += len(pi)
            if tp == 0:
                per_alpha.append(0.0); per_alpha_det.append(0.0); per_alpha_ass.append(0.0)
                continue
            ass_den = gt_count[:, None] + pr_count[None, :] - matches
            ass_iou = matches / np.maximum(ass_den, EPS)
            a_ass = float((matches * ass_iou).sum() / tp)
            a_det = float(tp / (tp + fn + fp))
            per_alpha_det.append(a_det)
            per_alpha_ass.append(a_ass)
            per_alpha.append(math.sqrt(a_det * a_ass))
        hota = float(np.mean(per_alpha))
        det_a = float(np.mean(per_alpha_det))
        ass_a = float(np.mean(per_alpha_ass))
        # HOTA at alpha=0.5 is reported too (easier to reason about)
        k = int(np.argmin(np.abs(ALPHAS - 0.5)))
        hota_at_half = float(per_alpha[k])

    # ---------------------------------------------------------------- IDF1
    if ng == 0 or npr == 0:
        idf1 = idp = idr = 0.0
        id_match = {}
    else:
        fn_mat = gt_count[:, None] - cooc_at_half + np.zeros((ng, npr))
        fp_mat = pr_count[None, :] - cooc_at_half + np.zeros((ng, npr))
        r, c = linear_sum_assignment(fn_mat + fp_mat)
        idtp = float(cooc_at_half[r, c].sum())
        idfn = n_gt_dets - idtp
        idfp = n_pr_dets - idtp
        idf1 = idtp / max(idtp + 0.5 * idfn + 0.5 * idfp, EPS)
        idp = idtp / max(idtp + idfp, EPS)
        idr = idtp / max(idtp + idfn, EPS)
        # one-to-one GT id <-> pred id assignment, used by the dashboard metrics
        id_match = {glist[gg]: plist[pp] for gg, pp in zip(r, c)
                    if cooc_at_half[gg, pp] > 0}

    # ------------------------------------------------------ CLEAR / MOTA / IDSW
    thr = CFG["eval_iou_threshold"]
    prev_pred_of_gt = {}
    tp = fp = fn = idsw = 0
    sim_sum = 0.0
    idsw_per_gt = collections.Counter()
    preds_per_gt = collections.defaultdict(set)
    for gi, pi, sim in cache:
        if len(gi) and len(pi):
            prev = np.array([[1.0 if prev_pred_of_gt.get(int(g), -1) == int(p) else 0.0
                              for p in pi] for g in gi])
            score = 1000.0 * prev + sim
            score[sim < thr - EPS] = 0.0
            r, c = linear_sum_assignment(-score)
            ok = score[r, c] > 0.0 + EPS
            r, c = r[ok], c[ok]
            for ri, ci in zip(r, c):
                g, p = int(gi[ri]), int(pi[ci])
                old = prev_pred_of_gt.get(g, None)
                if old is not None and old != p:
                    idsw += 1
                    idsw_per_gt[g] += 1
                prev_pred_of_gt[g] = p
                preds_per_gt[g].add(p)
            tp += len(r)
            sim_sum += float(sim[r, c].sum())
            fn += len(gi) - len(r)
            fp += len(pi) - len(c)
        else:
            fn += len(gi)
            fp += len(pi)
    mota = 1.0 - (fn + fp + idsw) / max(n_gt_dets, EPS)
    motp = sim_sum / max(tp, EPS)      # mean IoU over true positives
    recall = tp / max(n_gt_dets, EPS)
    precision = tp / max(n_pr_dets, EPS)

    inv_g = {v: k for k, v in gmap.items()}
    return {
        "hota": hota, "det_a": det_a, "ass_a": ass_a, "hota_50": hota_at_half,
        "idf1": idf1, "idp": idp, "idr": idr,
        "mota": mota, "motp": motp, "recall": recall, "precision": precision,
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "idsw": int(idsw),
        "n_gt_dets": int(n_gt_dets), "n_pred_dets": int(n_pr_dets),
        "n_gt_ids": ng, "n_pred_ids": npr,
        "idsw_per_1k_gt_dets": 1000.0 * idsw / max(n_gt_dets, EPS),
        "idsw_per_gt_id": idsw / max(ng, 1),
        "_id_match": id_match,
        "_idsw_per_gt": {int(inv_g[g]): int(v) for g, v in idsw_per_gt.items()},
        "_npred_per_gt": {int(inv_g[g]): len(v) for g, v in preds_per_gt.items()},
    }


print("cell 5 ok - alphas:", np.round(ALPHAS, 2))


## 6. Metric self-tests

A metric bug and a weak tracker look identical in the final table, so the run refuses to continue unless synthetic tracks with known answers come out right.

In [ ]:
# =====================================================================
# CELL 6 - self-tests on synthetic tracks with KNOWN answers.
# A metric bug here would look exactly like a weak tracker later, so the
# run refuses to continue unless these pass.
# =====================================================================
def _mk(tracks, n_frames):
    """tracks: {id: {frame: (x,y,w,h)}} -> frame -> {"ids","boxes"}."""
    out = {}
    for f in range(1, n_frames + 1):
        ids, boxes = [], []
        for tid, fr in tracks.items():
            if f in fr:
                x, y, w, h = fr[f]
                ids.append(tid)
                boxes.append((x, y, x + w, y + h))
        out[f] = {"ids": np.array(ids, np.int64),
                  "boxes": np.array(boxes, np.float64).reshape(-1, 4)}
    return out


NF = 100
# two people walking right, 40 px apart in y so they never overlap
A = {f: (10.0 + f, 10.0, 20.0, 40.0) for f in range(1, NF + 1)}
B = {f: (10.0 + f, 100.0, 20.0, 40.0) for f in range(1, NF + 1)}
GT2 = _mk({1: A, 2: B}, NF)

tests = []

# --- 1. perfect tracker -> everything 1.0, no ID switches
r = eval_sequence(GT2, _mk({7: A, 8: B}, NF), NF)
tests.append(("perfect hota", r["hota"], 1.0))
tests.append(("perfect idf1", r["idf1"], 1.0))
tests.append(("perfect mota", r["mota"], 1.0))
tests.append(("perfect idsw", r["idsw"], 0))
tests.append(("perfect det_a", r["det_a"], 1.0))
tests.append(("perfect ass_a", r["ass_a"], 1.0))

# --- 2. ids swapped at the halfway point -> 2 ID switches, IDF1 = 0.5
sw1 = dict(list(A.items())[:50] + list(B.items())[50:])
sw2 = dict(list(B.items())[:50] + list(A.items())[50:])
r = eval_sequence(GT2, _mk({7: sw1, 8: sw2}, NF), NF)
tests.append(("swap idsw", r["idsw"], 2))
tests.append(("swap idf1", r["idf1"], 0.5))
tests.append(("swap mota", r["mota"], 1.0 - 2.0 / 200.0))
tests.append(("swap det_a", r["det_a"], 1.0))          # detection is perfect
tests.append(("swap ass_a", r["ass_a"], 1.0 / 3.0))     # 50 of 150 union frames

# --- 3. every track cut into 10 fragments with a fresh id each time
frag = {}
for i in range(10):
    lo, hi = i * 10 + 1, (i + 1) * 10
    frag[100 + i] = {f: A[f] for f in range(lo, hi + 1)}
    frag[200 + i] = {f: B[f] for f in range(lo, hi + 1)}
r = eval_sequence(GT2, _mk(frag, NF), NF)
tests.append(("frag idsw", r["idsw"], 18))             # 9 switches per person
tests.append(("frag det_a", r["det_a"], 1.0))
tests.append(("frag idf1", r["idf1"], 0.1))            # best single id covers 10%
tests.append(("frag ass_a", r["ass_a"], 0.1))

# --- 4. one person missed entirely -> recall 0.5
r = eval_sequence(GT2, _mk({7: A}, NF), NF)
tests.append(("halfmiss recall", r["recall"], 0.5))
tests.append(("halfmiss det_a", r["det_a"], 0.5))
tests.append(("halfmiss mota", r["mota"], 0.5))
tests.append(("halfmiss ass_a", r["ass_a"], 1.0))      # the track it does hold is clean
tests.append(("halfmiss idf1", r["idf1"], 100.0 / (100.0 + 0.5 * 100.0)))

# --- 5. a pure duplicate of every box -> 100% spurious FPs
dup = {7: A, 8: B, 9: dict(A), 10: dict(B)}
r = eval_sequence(GT2, _mk(dup, NF), NF)
tests.append(("dup precision", r["precision"], 0.5))
tests.append(("dup mota", r["mota"], 0.0))             # FP == n_gt
tests.append(("dup fp", r["fp"], 200))

# --- 6. empty prediction -> zeros, not a crash
r = eval_sequence(GT2, _mk({}, NF), NF)
tests.append(("empty hota", r["hota"], 0.0))
tests.append(("empty idf1", r["idf1"], 0.0))
tests.append(("empty mota", r["mota"], 0.0))

bad = []
print("%-20s %12s %12s" % ("test", "got", "expected"))
for name, got, exp in tests:
    ok = abs(float(got) - float(exp)) < 1e-6
    print("%-20s %12.6f %12.6f  %s" % (name, got, exp, "ok" if ok else "FAIL"))
    if not ok:
        bad.append(name)
assert not bad, "metric self-tests failed: %s -- fix the metric before running" % bad
print("\nall %d metric self-tests passed" % len(tests))


## 7. Tracker

ByteTrack-style association. Five configs of deliberately varying association strength, so the ID-switch rate can be varied while the detections stay byte-identical.

In [ ]:
# =====================================================================
# CELL 7 - ByteTrack-style tracker, written here (nothing is trained).
# The association strength is configurable so that several trackers can be
# run over IDENTICAL detections. That holds detection quality constant and
# lets the ID-switch rate be the only thing that varies - which is exactly
# what the dashboard-metric comparison needs.
# =====================================================================
class KalmanBoxFilter(object):
    """Constant-velocity filter on (cx, cy, aspect, height), ByteTrack's form.
    Process and measurement noise scale with box height, so far-away (small)
    people are tracked with proportionally tighter gates."""

    def __init__(self):
        self.F = np.eye(8)
        for i in range(4):
            self.F[i, 4 + i] = 1.0
        self.H = np.eye(4, 8)
        self.wp = 1.0 / 20.0
        self.wv = 1.0 / 160.0

    def initiate(self, z):
        mean = np.r_[z, np.zeros(4)]
        h = z[3]
        std = np.array([2 * self.wp * h, 2 * self.wp * h, 1e-2, 2 * self.wp * h,
                        10 * self.wv * h, 10 * self.wv * h, 1e-5, 10 * self.wv * h])
        return mean, np.diag(std ** 2)

    def predict(self, mean, cov):
        h = mean[3]
        std = np.array([self.wp * h, self.wp * h, 1e-2, self.wp * h,
                        self.wv * h, self.wv * h, 1e-5, self.wv * h])
        mean = self.F.dot(mean)
        cov = self.F.dot(cov).dot(self.F.T) + np.diag(std ** 2)
        return mean, cov

    def update(self, mean, cov, z):
        h = mean[3]
        std = np.array([self.wp * h, self.wp * h, 1e-1, self.wp * h])
        pm = self.H.dot(mean)
        pc = self.H.dot(cov).dot(self.H.T) + np.diag(std ** 2)
        ct = cov.dot(self.H.T)
        gain = np.linalg.solve(pc.T, ct.T).T
        mean = mean + gain.dot(z - pm)
        cov = cov - gain.dot(pc).dot(gain.T)
        return mean, cov


KF = KalmanBoxFilter()


def xyxy_to_cxah(b):
    x1, y1, x2, y2 = b
    w, h = max(x2 - x1, 1e-3), max(y2 - y1, 1e-3)
    return np.array([x1 + w / 2.0, y1 + h / 2.0, w / h, h], dtype=np.float64)


def cxah_to_xyxy(m):
    cx, cy, a, h = m[0], m[1], m[2], m[3]
    w = max(a, 1e-3) * max(h, 1e-3)
    return np.array([cx - w / 2.0, cy - h / 2.0, cx + w / 2.0, cy + h / 2.0])


class Track(object):
    __slots__ = ("tid", "box", "mean", "cov", "score", "state", "hits",
                 "last_frame", "start_frame", "activated")

    def __init__(self, tid, box, score, frame, use_kalman):
        self.tid = tid
        self.box = np.asarray(box, np.float64)
        self.score = score
        self.state = "tracked"
        self.hits = 1
        self.last_frame = frame
        self.start_frame = frame
        self.activated = False
        if use_kalman:
            self.mean, self.cov = KF.initiate(xyxy_to_cxah(box))
        else:
            self.mean, self.cov = None, None

    def predict(self):
        if self.mean is not None:
            if self.state != "tracked":
                self.mean[7] = 0.0          # a lost track stops growing/shrinking
            self.mean, self.cov = KF.predict(self.mean, self.cov)
            self.box = cxah_to_xyxy(self.mean)

    def update(self, box, score, frame):
        self.box = np.asarray(box, np.float64)
        self.score = score
        self.hits += 1
        self.last_frame = frame
        self.state = "tracked"
        self.activated = True
        if self.mean is not None:
            self.mean, self.cov = KF.update(self.mean, self.cov, xyxy_to_cxah(box))
            self.box = cxah_to_xyxy(self.mean)


def _assign(tracks, boxes, max_dist):
    """Hungarian on IoU distance, rejecting pairs worse than max_dist."""
    if not tracks or len(boxes) == 0:
        return [], list(range(len(tracks))), list(range(len(boxes)))
    cost = 1.0 - iou_matrix([t.box for t in tracks], boxes)
    r, c = linear_sum_assignment(cost)
    matches, ur, uc = [], set(range(len(tracks))), set(range(len(boxes)))
    for ri, ci in zip(r, c):
        if cost[ri, ci] <= max_dist:
            matches.append((int(ri), int(ci)))
            ur.discard(int(ri))
            uc.discard(int(ci))
    return matches, sorted(ur), sorted(uc)


TRACKER_CONFIGS = json.loads(r"""
{
  "bytetrack":  {"kalman": true,  "two_stage": true,  "match": 0.80, "second_match": 0.50,
                 "buffer": 30, "min_hits": 2, "high": 0.50, "new_track": 0.60},
  "no-recovery":{"kalman": true,  "two_stage": false, "match": 0.80, "second_match": 0.50,
                 "buffer": 30, "min_hits": 2, "high": 0.50, "new_track": 0.60},
  "short-memory":{"kalman": true, "two_stage": false, "match": 0.70, "second_match": 0.50,
                 "buffer": 5,  "min_hits": 2, "high": 0.50, "new_track": 0.60},
  "iou-only":   {"kalman": false, "two_stage": false, "match": 0.70, "second_match": 0.50,
                 "buffer": 3,  "min_hits": 2, "high": 0.50, "new_track": 0.60},
  "memoryless": {"kalman": false, "two_stage": false, "match": 0.50, "second_match": 0.50,
                 "buffer": 0,  "min_hits": 1, "high": 0.50, "new_track": 0.60}
}
""")
PRIMARY_TRACKER = "bytetrack"


def run_tracker(dets_by_frame, n_frames, cfg):
    """dets_by_frame: frame -> (boxes (N,4), scores (N,)).
    Returns frame -> {"ids","boxes"} with anonymous integer ids."""
    active, lost, out = [], [], {}
    next_id = 1
    for fr in range(1, n_frames + 1):
        boxes, scores = dets_by_frame.get(fr, (np.zeros((0, 4)), np.zeros(0)))
        boxes = np.asarray(boxes, np.float64).reshape(-1, 4)
        scores = np.asarray(scores, np.float64).reshape(-1)
        hi = scores >= cfg["high"]
        if cfg["two_stage"]:
            b_hi, s_hi = boxes[hi], scores[hi]
            lo = (~hi) & (scores >= CFG["det_score_keep"])
            b_lo, s_lo = boxes[lo], scores[lo]
        else:
            b_hi, s_hi = boxes[hi], scores[hi]
            b_lo, s_lo = np.zeros((0, 4)), np.zeros(0)

        for t in active + lost:
            t.predict()

        confirmed = [t for t in active if t.activated]
        tentative = [t for t in active if not t.activated]
        pool = confirmed + lost
        m, u_t, u_d = _assign(pool, b_hi, cfg["match"])
        for ti, di in m:
            pool[ti].update(b_hi[di], s_hi[di], fr)
        matched_pool = set(ti for ti, _ in m)

        # stage two: recover tracks using the LOW-confidence detections
        if cfg["two_stage"] and len(b_lo):
            rem = [i for i in u_t if pool[i].state == "tracked"]
            m2, u_t2, _ = _assign([pool[i] for i in rem], b_lo, cfg["second_match"])
            for ti, di in m2:
                pool[rem[ti]].update(b_lo[di], s_lo[di], fr)
                matched_pool.add(rem[ti])

        # tentative tracks compete for the leftover high-score detections
        left = [i for i in u_d]
        m3, u_t3, u_d3 = _assign(tentative, b_hi[left] if left else np.zeros((0, 4)), 0.70)
        for ti, di in m3:
            tentative[ti].update(b_hi[left[di]], s_hi[left[di]], fr)
        matched_tent = set(ti for ti, _ in m3)
        leftover = [left[i] for i in u_d3]

        new_active, new_lost = [], []
        for i, t in enumerate(pool):
            if i in matched_pool:
                new_active.append(t)
            elif fr - t.last_frame <= cfg["buffer"]:
                t.state = "lost"
                new_lost.append(t)
        for i, t in enumerate(tentative):
            if i in matched_tent:
                new_active.append(t)
            # an unmatched tentative track is dropped immediately (ByteTrack rule)
        for di in leftover:
            if s_hi[di] >= cfg["new_track"]:
                nt = Track(next_id, b_hi[di], s_hi[di], fr, cfg["kalman"])
                next_id += 1
                if cfg["min_hits"] <= 1 or fr == 1:
                    nt.activated = True
                    new_active.append(nt)
                else:
                    new_active.append(nt)
        active, lost = new_active, new_lost

        ids = [t.tid for t in active if t.activated and t.last_frame == fr]
        bxs = [t.box for t in active if t.activated and t.last_frame == fr]
        out[fr] = {"ids": np.array(ids, np.int64),
                   "boxes": np.array(bxs, np.float64).reshape(-1, 4)}
    return out


print("cell 7 ok - tracker configs:", list(TRACKER_CONFIGS))


## 8. Person detection

COCO-pretrained Faster R-CNN, plus the benchmark's own public detections as a second source that costs no GPU time.

In [ ]:
# =====================================================================
# CELL 8 - person detection. A COCO-pretrained detector off the shelf; no
# training, no fine-tuning on MOT17, no GPU time spent on weights.
# =====================================================================
import torchvision
from torchvision.models.detection import (fasterrcnn_resnet50_fpn_v2,
                                          FasterRCNN_ResNet50_FPN_V2_Weights)
from torch.utils.data import Dataset, DataLoader
from PIL import Image

COCO_PERSON = 1


def build_detector():
    w = FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1
    model = fasterrcnn_resnet50_fpn_v2(
        weights=w, box_score_thresh=CFG["det_score_keep"],
        min_size=CFG["det_min_size"], max_size=CFG["det_max_size"])
    model.eval().to(DEV)
    n = sum(p.numel() for p in model.parameters())
    print("detector: fasterrcnn_resnet50_fpn_v2 COCO_V1, %.1fM params, "
          "min_size=%d max_size=%d" % (n / 1e6, CFG["det_min_size"], CFG["det_max_size"]))
    return model


class FrameSet(Dataset):
    def __init__(self, seqdir, n_frames, ext):
        self.dir = os.path.join(seqdir, "img1")
        self.names = sorted(os.listdir(self.dir))
        self.n = n_frames
        self.ext = ext

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        fr = i + 1
        p = os.path.join(self.dir, "%06d%s" % (fr, self.ext))
        if not os.path.exists(p):
            p = os.path.join(self.dir, self.names[i])
        img = Image.open(p).convert("RGB")
        t = torch.from_numpy(np.asarray(img, dtype=np.uint8)).permute(2, 0, 1)
        return fr, t.float().div_(255.0)


def collate(batch):
    return [b[0] for b in batch], [b[1] for b in batch]


@torch.no_grad()
def detect_sequence(model, seq, batch_size=4, log_every=200):
    """frame -> (boxes (N,4) xyxy, scores (N,)) for the person class only."""
    seqdir = pick_variant(seq, CFG["gt_variant"])
    info = SEQINFO[seq]
    ds = FrameSet(seqdir, info["n_frames"], info["imext"])
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2,
                    collate_fn=collate, pin_memory=False)
    out = {}
    t0 = time.time()
    done = 0
    for frames, imgs in dl:
        preds = model([im.to(DEV, non_blocking=True) for im in imgs])
        for fr, p in zip(frames, preds):
            keep = p["labels"] == COCO_PERSON
            b = p["boxes"][keep].detach().cpu().numpy().astype(np.float64).reshape(-1, 4)
            s = p["scores"][keep].detach().cpu().numpy().astype(np.float64).reshape(-1)
            out[int(fr)] = (b, s)
        done += len(frames)
        if done % log_every < batch_size:
            el = time.time() - t0
            print("    %s %4d/%4d frames  %.3f s/frame  eta %.1f min"
                  % (seq, done, info["n_frames"], el / max(done, 1),
                     (info["n_frames"] - done) * el / max(done, 1) / 60.0), flush=True)
    el = time.time() - t0
    nd = sum(len(v[0]) for v in out.values())
    print("  %s: %d frames, %d person detections, %.1f s (%.3f s/frame)"
          % (seq, len(out), nd, el, el / max(len(out), 1)), flush=True)
    assert len(out) == info["n_frames"], "detected %d of %d frames" % (len(out), info["n_frames"])
    assert nd > 0, "detector produced ZERO person detections on " + seq
    return out, el


def public_detections(seq):
    """The benchmark's own public detections, as a second detection source
    that costs no GPU time at all."""
    seqdir = pick_variant(seq, CFG["public_det_variant"])
    raw = read_mot_txt(os.path.join(seqdir, "det", "det.txt"), is_gt=False)
    scores = np.array([r[2] for f in raw for r in raw[f]], dtype=np.float64)
    lo, hi = float(scores.min()), float(scores.max())
    out = {}
    for fr in range(1, SEQINFO[seq]["n_frames"] + 1):
        rows = raw.get(fr, [])
        b = np.array([r[1] for r in rows], np.float64).reshape(-1, 4)
        s = np.array([r[2] for r in rows], np.float64).reshape(-1)
        if hi > 1.0 or lo < 0.0:            # DPM/SDP ship unnormalised confidences
            s = (s - lo) / max(hi - lo, 1e-9)
        out[fr] = (b, s)
    print("  %s public (%s): %d detections, raw score range [%.3f, %.3f]"
          % (seq, os.path.basename(seqdir), len(scores), lo, hi))
    return out


print("cell 8 ok")


## 9. The dashboard numbers

Each one computed identically for ground truth and for the tracker, so each has a true value to be wrong about.

In [ ]:
# =====================================================================
# CELL 9 - the "dashboard" numbers, computed identically for ground truth
# and for the tracker output, so every one of them has a true value.
#
#   dwell time      per identity  -> needs the identity held for the visit
#   zone occupancy  per frame     -> needs detection only, no identity
#   line crossings  per identity  -> needs the identity held across the line
#   line crossings  total         -> aggregate, identity-free
#
# "times this person crossed the counter line" is the measurable stand-in
# for "cups made by this person": same shape of claim, same dependence on
# holding one identity for a whole visit, but it has a ground truth.
# =====================================================================
def tracks_from_frames(fr_dict, n_frames):
    """frame -> {"ids","boxes"}  =>  id -> {"frames":[...], "boxes":[...]}"""
    tr = collections.defaultdict(lambda: {"frames": [], "boxes": []})
    for fr in range(1, n_frames + 1):
        d = fr_dict.get(fr, EMPTY)
        for i, b in zip(d["ids"], np.asarray(d["boxes"]).reshape(-1, 4)):
            tr[int(i)]["frames"].append(fr)
            tr[int(i)]["boxes"].append(b)
    return {k: {"frames": v["frames"], "boxes": np.array(v["boxes"]).reshape(-1, 4)}
            for k, v in tr.items()}


def dwell_seconds(track, fps):
    """Span from first to last sighting - what a dashboard calls time-in-store."""
    f = track["frames"]
    return (f[-1] - f[0] + 1) / float(fps)


def zone_occupancy(fr_dict, n_frames, w, h):
    """Distinct identities whose feet fall inside the zone, per frame."""
    x0, x1 = CFG["zone_x"][0] * w, CFG["zone_x"][1] * w
    y0, y1 = CFG["zone_y"][0] * h, CFG["zone_y"][1] * h
    occ = np.zeros(n_frames, dtype=np.float64)
    for fr in range(1, n_frames + 1):
        d = fr_dict.get(fr, EMPTY)
        b = np.asarray(d["boxes"]).reshape(-1, 4)
        if len(b) == 0:
            continue
        fx = (b[:, 0] + b[:, 2]) / 2.0        # foot point: bottom-centre of the box
        fy = b[:, 3]
        inside = (fx >= x0) & (fx <= x1) & (fy >= y0) & (fy <= y1)
        occ[fr - 1] = float(np.count_nonzero(inside))
    return occ


def zone_occupancy_from_dets(dets, n_frames, w, h, score_min):
    """Occupancy counted straight from the DETECTIONS - no tracker, no
    identities at all. This is the honest identity-free baseline: whatever
    error it has is pure detection error."""
    x0, x1 = CFG["zone_x"][0] * w, CFG["zone_x"][1] * w
    y0, y1 = CFG["zone_y"][0] * h, CFG["zone_y"][1] * h
    occ = np.zeros(n_frames, dtype=np.float64)
    for fr in range(1, n_frames + 1):
        b, s = dets.get(fr, (np.zeros((0, 4)), np.zeros(0)))
        b = np.asarray(b, np.float64).reshape(-1, 4)
        s = np.asarray(s, np.float64).reshape(-1)
        if len(b) == 0:
            continue
        keep = s >= score_min
        b = b[keep]
        if len(b) == 0:
            continue
        fx = (b[:, 0] + b[:, 2]) / 2.0
        fy = b[:, 3]
        occ[fr - 1] = float(np.count_nonzero((fx >= x0) & (fx <= x1)
                                             & (fy >= y0) & (fy <= y1)))
    return occ


def line_crossings(track, w):
    """How many times this identity's foot point crossed the vertical line."""
    lx = CFG["line_x_frac"] * w
    b = track["boxes"]
    if len(b) < 2:
        return 0
    cx = (b[:, 0] + b[:, 2]) / 2.0
    side = np.sign(cx - lx)
    side = side[side != 0]
    if len(side) < 2:
        return 0
    return int(np.count_nonzero(side[1:] != side[:-1]))


def dashboard_for(fr_dict, gt):
    n, w, h, fps = gt["n_frames"], gt["width"], gt["height"], gt["fps"]
    tr = tracks_from_frames(fr_dict, n)
    return {"tracks": tr,
            "dwell": {k: dwell_seconds(v, fps) for k, v in tr.items()},
            "cross": {k: line_crossings(v, w) for k, v in tr.items()},
            "occ": zone_occupancy(fr_dict, n, w, h)}


def dashboard_errors(gt, gt_frames, pred_frames, id_match, idsw_per_gt, npred_per_gt,
                     dets=None, det_score_min=0.5):
    """Every dashboard number, measured against its ground truth."""
    G = dashboard_for(gt_frames, gt)
    P = dashboard_for(pred_frames, gt)
    fps = gt["fps"]

    # ---- zone occupancy: an aggregate count, no identity required
    occ_err = P["occ"] - G["occ"]
    occ = {"mae": float(np.mean(np.abs(occ_err))),
           "bias": float(np.mean(occ_err)),
           "true_mean": float(np.mean(G["occ"])),
           "pred_mean": float(np.mean(P["occ"])),
           "mae_pct_of_mean": float(100.0 * np.mean(np.abs(occ_err))
                                    / max(np.mean(G["occ"]), 1e-9)),
           "frames": int(len(occ_err))}

    # ---- the same aggregate, with no tracker in the loop at all
    occ_det = None
    if dets is not None:
        od = zone_occupancy_from_dets(dets, gt["n_frames"], gt["width"],
                                      gt["height"], det_score_min)
        e = od - G["occ"]
        occ_det = {"mae": float(np.mean(np.abs(e))), "bias": float(np.mean(e)),
                   "pred_mean": float(np.mean(od)),
                   "true_mean": float(np.mean(G["occ"])),
                   "mae_pct_of_mean": float(100.0 * np.mean(np.abs(e))
                                            / max(np.mean(G["occ"]), 1e-9)),
                   "frames": int(len(e)), "score_min": float(det_score_min)}

    # ---- per-identity dwell and crossings, over the optimal GT<->pred pairing
    rows = []
    for g, tg in G["tracks"].items():
        p = id_match.get(g, None)
        dt = G["dwell"][g]
        dp = P["dwell"].get(p, 0.0) if p is not None else 0.0
        ct = G["cross"][g]
        cp = P["cross"].get(p, 0) if p is not None else 0
        rows.append({"gt_id": int(g), "matched": p is not None,
                     "dwell_true": dt, "dwell_pred": dp,
                     "dwell_abs_err": abs(dp - dt),
                     "dwell_pct_err": 100.0 * abs(dp - dt) / max(dt, 1e-9),
                     "cross_true": int(ct), "cross_pred": int(cp),
                     "cross_abs_err": abs(int(cp) - int(ct)),
                     "idsw": int(idsw_per_gt.get(int(g), 0)),
                     "n_pred_ids": int(npred_per_gt.get(int(g), 0)),
                     "gt_frames": len(tg["frames"])})
    matched = [r for r in rows if r["matched"]]

    def agg(rs, key):
        return float(np.mean([r[key] for r in rs])) if rs else float("nan")

    dwell = {
        "n_gt_identities": len(rows),
        "n_matched_identities": len(matched),
        "mae_sec_matched": agg(matched, "dwell_abs_err"),
        "mape_pct_matched": agg(matched, "dwell_pct_err"),
        "mae_sec_all": agg(rows, "dwell_abs_err"),
        "mape_pct_all": agg(rows, "dwell_pct_err"),
        "median_pct_err_matched": (float(np.median([r["dwell_pct_err"] for r in matched]))
                                   if matched else float("nan")),
        "true_mean_sec": agg(rows, "dwell_true"),
        "pred_mean_sec_matched": agg(matched, "dwell_pred"),
    }

    # ---- what the dashboard would actually display: one row per TRACK,
    # because it has no way to know two tracks were the same person.
    pred_dwells = [P["dwell"][k] for k in P["tracks"]]
    naive = {
        "true_n_visitors": len(rows),
        "reported_n_visitors": len(pred_dwells),
        "visitor_count_pct_err": 100.0 * (len(pred_dwells) - len(rows)) / max(len(rows), 1),
        "true_mean_dwell_sec": agg(rows, "dwell_true"),
        "reported_mean_dwell_sec": float(np.mean(pred_dwells)) if pred_dwells else 0.0,
        "mean_dwell_pct_err": (100.0 * (float(np.mean(pred_dwells)) - agg(rows, "dwell_true"))
                               / max(agg(rows, "dwell_true"), 1e-9)) if pred_dwells else -100.0,
    }

    # ---- crossings: per-person versus the same quantity summed over everyone
    tot_t = int(sum(G["cross"].values()))
    tot_p = int(sum(P["cross"].values()))
    cross = {
        "per_person_mae": agg(matched, "cross_abs_err"),
        "per_person_mae_all": agg(rows, "cross_abs_err"),
        "per_person_exact_frac": (float(np.mean([r["cross_abs_err"] == 0 for r in matched]))
                                  if matched else float("nan")),
        "total_true": tot_t, "total_pred": tot_p,
        "total_abs_err": abs(tot_p - tot_t),
        "total_pct_err": 100.0 * abs(tot_p - tot_t) / max(tot_t, 1),
    }
    return {"occupancy": occ, "occupancy_dets_only": occ_det, "dwell": dwell,
            "dwell_naive": naive, "crossings": cross, "_rows": rows}


print("cell 9 ok - zone x%s y%s, counter line at x=%.2fW"
      % (CFG["zone_x"], CFG["zone_y"], CFG["line_x_frac"]))


## 10. Run

In [ ]:
# =====================================================================
# CELL 10 - THE RUN. Detections are computed once per sequence and reused
# by every tracker config, so the comparison varies association only.
# =====================================================================
SEQS_TO_RUN = CFG["dev_sequences"][:1] if PILOT_ONLY else ALL_SEQS
print("PILOT_ONLY =", PILOT_ONLY, "-> sequences:", SEQS_TO_RUN)
print("frames to detect:", sum(SEQINFO[s]["n_frames"] for s in SEQS_TO_RUN))

GT_CACHE, RUNS, DET_STATS = {}, [], []
GPU_SECONDS = 0.0
BUDGET_SECONDS = 5.5 * 3600          # hard stop well inside the 9 h kernel limit

MODEL = build_detector()

for seq in SEQS_TO_RUN:
    print("\n================ %s ================" % seq, flush=True)
    gt = load_gt(seq)
    GT_CACHE[seq] = gt
    print("  gt: %d frames, %d pedestrian boxes, %d identities, %.0f fps, %dx%d"
          % (gt["n_frames"], gt["n_gt_dets"], gt["n_gt_ids"], gt["fps"],
             gt["width"], gt["height"]))
    assert gt["n_gt_dets"] > 0 and gt["n_gt_ids"] > 0, "empty ground truth for " + seq

    if time.time() - T_START > BUDGET_SECONDS:
        print("  !! time budget reached, stopping before", seq)
        break

    sources = {}
    d, el = detect_sequence(MODEL, seq)
    GPU_SECONDS += el
    sources["frcnn-v2"] = d
    sources["public-" + CFG["public_det_variant"].lower()] = public_detections(seq)

    for dname, dets in sources.items():
        nd = sum(len(v[0]) for v in dets.values())
        hi = sum(int((v[1] >= 0.5).sum()) for v in dets.values())
        DET_STATS.append({"seq": seq, "det": dname, "n_dets": nd, "n_above_half": hi,
                          "per_frame": nd / float(gt["n_frames"])})
        print("  [%s] %d dets (%d with score>=0.5), %.1f per frame"
              % (dname, nd, hi, nd / float(gt["n_frames"])), flush=True)

        for cfgname, tcfg in TRACKER_CONFIGS.items():
            t0 = time.time()
            pred = run_tracker(dets, gt["n_frames"], tcfg)
            pred_pp, removed = preprocess_predictions(gt, pred)
            m = eval_sequence(gt["eval"], pred_pp, gt["n_frames"])
            dash = dashboard_errors(gt, gt["eval"], pred_pp, m["_id_match"],
                                    m["_idsw_per_gt"], m["_npred_per_gt"],
                                    dets=dets, det_score_min=tcfg["high"])
            rec = {"seq": seq, "det": dname, "tracker": cfgname,
                   "split": "dev" if seq in CFG["dev_sequences"] else "test",
                   "distractor_dets_removed": removed,
                   "eval_seconds": time.time() - t0}
            for k, v in m.items():
                if not k.startswith("_"):
                    rec[k] = v
            rec["occupancy"] = dash["occupancy"]
            rec["occupancy_dets_only"] = dash["occupancy_dets_only"]
            rec["dwell"] = dash["dwell"]
            rec["dwell_naive"] = dash["dwell_naive"]
            rec["crossings"] = dash["crossings"]
            rec["_rows"] = dash["_rows"]
            RUNS.append(rec)
            print("    %-13s HOTA %.4f IDF1 %.4f MOTA %.4f IDSW %4d | "
                  "dwell MAPE %6.1f%% occ MAE %.3f cross/person MAE %.3f | %.0fs"
                  % (cfgname, m["hota"], m["idf1"], m["mota"], m["idsw"],
                     dash["dwell"]["mape_pct_matched"], dash["occupancy"]["mae"],
                     dash["crossings"]["per_person_mae"], rec["eval_seconds"]), flush=True)

assert RUNS, "the run produced NO results - refusing to report anything"
print("\n==== run finished: %d (sequence x detector x tracker) results ====" % len(RUNS))
print("GPU detection time: %.1f min over %d frames"
      % (GPU_SECONDS / 60.0, sum(SEQINFO[s]["n_frames"] for s in GT_CACHE)))
print("wall clock so far : %.1f min" % ((time.time() - T_START) / 60.0))

if PILOT_ONLY:
    nf_pilot = sum(SEQINFO[s]["n_frames"] for s in GT_CACHE)
    nf_full = sum(SEQINFO[s]["n_frames"] for s in ALL_SEQS)
    per = GPU_SECONDS / max(nf_pilot, 1)
    wall = (time.time() - T_START) / max(nf_pilot, 1)
    print("\n---- COST EXTRAPOLATION ----")
    print("  %.3f s/frame detection -> full set (%d frames) = %.1f min GPU"
          % (per, nf_full, per * nf_full / 60.0))
    print("  %.1f min wall per 1000 frames -> full set approx %.1f min wall"
          % (wall * 1000 / 60.0, wall * nf_full / 60.0))


## 11. Does the error follow the ID-switch rate?

In [ ]:
# =====================================================================
# CELL 11 - the central comparison: which dashboard numbers degrade with
# the ID-switch rate, and which do not.
# =====================================================================
from scipy.stats import pearsonr, spearmanr


def pool(rows):
    """Aggregate a set of per-sequence runs that share detector + tracker."""
    o = {"n_sequences": len(rows), "sequences": sorted(r["seq"] for r in rows)}
    for k in ("hota", "det_a", "ass_a", "idf1", "mota", "motp", "recall",
              "precision", "idsw_per_gt_id", "idsw_per_1k_gt_dets"):
        o[k] = float(np.mean([r[k] for r in rows]))        # macro mean over sequences
    gtd = float(sum(r["n_gt_dets"] for r in rows))
    o["n_gt_dets"] = int(gtd)
    o["n_gt_ids"] = int(sum(r["n_gt_ids"] for r in rows))
    o["idsw_total"] = int(sum(r["idsw"] for r in rows))
    o["fn_total"] = int(sum(r["fn"] for r in rows))
    o["fp_total"] = int(sum(r["fp"] for r in rows))
    o["mota_micro"] = float(1.0 - (o["fn_total"] + o["fp_total"] + o["idsw_total"]) / max(gtd, 1))
    o["idsw_per_gt_id_micro"] = float(o["idsw_total"] / max(o["n_gt_ids"], 1))

    ids = [x for r in rows for x in r["_rows"]]
    matched = [x for x in ids if x["matched"]]
    o["n_identities"] = len(ids)
    o["n_identities_matched"] = len(matched)

    def m(rs, k):
        return float(np.mean([x[k] for x in rs])) if rs else float("nan")

    # --- PER-PERSON numbers (need the identity held for the whole visit)
    o["dwell_mae_sec_matched"] = m(matched, "dwell_abs_err")
    o["dwell_mape_matched"] = m(matched, "dwell_pct_err")
    o["dwell_median_pct_matched"] = (float(np.median([x["dwell_pct_err"] for x in matched]))
                                     if matched else float("nan"))
    o["dwell_mae_sec_all"] = m(ids, "dwell_abs_err")
    o["dwell_mape_all"] = m(ids, "dwell_pct_err")
    o["dwell_true_mean_sec"] = m(ids, "dwell_true")
    o["cross_person_mae"] = m(matched, "cross_abs_err")
    ct_mean = m(ids, "cross_true")
    o["cross_true_mean_per_person"] = ct_mean
    o["cross_person_mae_pct"] = float(100.0 * o["cross_person_mae"] / max(ct_mean, 1e-9))
    o["cross_person_exact_frac"] = (float(np.mean([x["cross_abs_err"] == 0 for x in matched]))
                                    if matched else float("nan"))

    # --- AGGREGATE numbers (need detection only)
    nfr = float(sum(r["occupancy"]["frames"] for r in rows))
    abs_err = sum(r["occupancy"]["mae"] * r["occupancy"]["frames"] for r in rows)
    true_sum = sum(r["occupancy"]["true_mean"] * r["occupancy"]["frames"] for r in rows)
    pred_sum = sum(r["occupancy"]["pred_mean"] * r["occupancy"]["frames"] for r in rows)
    o["occ_mae"] = float(abs_err / max(nfr, 1))
    o["occ_true_mean"] = float(true_sum / max(nfr, 1))
    o["occ_pred_mean"] = float(pred_sum / max(nfr, 1))
    o["occ_mae_pct"] = float(100.0 * abs_err / max(true_sum, 1e-9))
    o["occ_bias"] = float((pred_sum - true_sum) / max(nfr, 1))
    if all(r.get("occupancy_dets_only") for r in rows):
        ae = sum(r["occupancy_dets_only"]["mae"] * r["occupancy_dets_only"]["frames"]
                 for r in rows)
        ps = sum(r["occupancy_dets_only"]["pred_mean"] * r["occupancy_dets_only"]["frames"]
                 for r in rows)
        o["occ_det_mae"] = float(ae / max(nfr, 1))
        o["occ_det_mae_pct"] = float(100.0 * ae / max(true_sum, 1e-9))
        o["occ_det_pred_mean"] = float(ps / max(nfr, 1))
        o["occ_det_bias"] = float((ps - true_sum) / max(nfr, 1))
    else:
        o["occ_det_mae"] = o["occ_det_mae_pct"] = float("nan")
        o["occ_det_pred_mean"] = o["occ_det_bias"] = float("nan")
    ctt = sum(r["crossings"]["total_true"] for r in rows)
    ctp = sum(r["crossings"]["total_pred"] for r in rows)
    o["cross_total_true"] = int(ctt)
    o["cross_total_pred"] = int(ctp)
    o["cross_total_pct_err"] = float(100.0 * abs(ctp - ctt) / max(ctt, 1))

    # --- what the dashboard would print, knowing nothing about fragmentation
    o["visitors_true"] = len(ids)
    o["visitors_reported"] = int(sum(r["dwell_naive"]["reported_n_visitors"] for r in rows))
    o["visitor_count_pct_err"] = float(100.0 * (o["visitors_reported"] - o["visitors_true"])
                                       / max(o["visitors_true"], 1))
    rep = [r["dwell_naive"]["reported_mean_dwell_sec"] * r["dwell_naive"]["reported_n_visitors"]
           for r in rows]
    o["reported_mean_dwell_sec"] = float(sum(rep) / max(o["visitors_reported"], 1))
    o["reported_mean_dwell_pct_err"] = float(
        100.0 * (o["reported_mean_dwell_sec"] - o["dwell_true_mean_sec"])
        / max(o["dwell_true_mean_sec"], 1e-9))
    return o


# ---------------------------------------------------------------- detector choice
DETS = sorted(set(r["det"] for r in RUNS))
dev_runs = [r for r in RUNS if r["split"] == "dev" and r["tracker"] == PRIMARY_TRACKER]
print("==== detector choice, decided on the DEV sequences only ====")
DEV_BY_DET = {}
for d in DETS:
    rs = [r for r in dev_runs if r["det"] == d]
    if rs:
        p = pool(rs)
        DEV_BY_DET[d] = p
        print("  %-16s dev IDF1 %.4f HOTA %.4f MOTA %.4f recall %.3f prec %.3f"
              % (d, p["idf1"], p["hota"], p["mota"], p["recall"], p["precision"]))
PRIMARY_DET = max(DEV_BY_DET, key=lambda d: DEV_BY_DET[d]["idf1"]) if DEV_BY_DET else DETS[0]
print("  -> primary detector:", PRIMARY_DET)

HAVE_TEST = any(r["split"] == "test" for r in RUNS)
SCOPE = "test" if HAVE_TEST else "dev"
scope_runs = [r for r in RUNS if r["split"] == SCOPE and r["det"] == PRIMARY_DET]
print("\nreporting scope: %s sequences (%s)"
      % (SCOPE, sorted(set(r["seq"] for r in scope_runs))))

# ---------------------------------------------------------------- headline table
print("\n==== %s: %s detections, every tracker config ====" % (SCOPE.upper(), PRIMARY_DET))
hdr = ("%-13s %6s %6s %6s %6s %7s | %8s %8s | %7s %7s | %8s %8s"
       % ("tracker", "HOTA", "DetA", "AssA", "IDF1", "MOTA", "IDSW", "IDSW/id",
          "dwell%", "dwellS", "occMAE%", "cross%"))
print(hdr)
print("-" * len(hdr))
print("  (occMAE% is occupancy measured off the TRACKER output; the no-tracker "
      "baseline is printed below the table)")
BY_TRACKER = {}
for cfgname in TRACKER_CONFIGS:
    rs = [r for r in scope_runs if r["tracker"] == cfgname]
    if not rs:
        continue
    p = pool(rs)
    BY_TRACKER[cfgname] = p
    print("%-13s %6.4f %6.4f %6.4f %6.4f %7.4f | %8d %8.2f | %7.1f %7.1f | %8.2f %8.2f"
          % (cfgname, p["hota"], p["det_a"], p["ass_a"], p["idf1"], p["mota"],
             p["idsw_total"], p["idsw_per_gt_id_micro"], p["dwell_mape_matched"],
             p["dwell_mae_sec_matched"], p["occ_mae_pct"], p["cross_total_pct_err"]))

PRIMARY = BY_TRACKER[PRIMARY_TRACKER]

# ---------------------------------------------------------------- per sequence
print("\n==== per sequence, %s + %s ====" % (PRIMARY_DET, PRIMARY_TRACKER))
print("%-11s %5s %6s %6s %6s %6s %6s %6s %7s %7s"
      % ("seq", "ids", "HOTA", "IDF1", "MOTA", "IDSW", "dwell%", "occMAE", "occMAE%", "xtot%"))
PER_SEQ = {}
for r in sorted([x for x in scope_runs if x["tracker"] == PRIMARY_TRACKER],
                key=lambda z: z["seq"]):
    PER_SEQ[r["seq"]] = {k: r[k] for k in
                         ("hota", "det_a", "ass_a", "idf1", "mota", "motp", "idsw",
                          "recall", "precision", "n_gt_ids", "n_gt_dets",
                          "idsw_per_gt_id")}
    PER_SEQ[r["seq"]].update({"occupancy": r["occupancy"], "dwell": r["dwell"],
                              "crossings": r["crossings"],
                              "dwell_naive": r["dwell_naive"]})
    print("%-11s %5d %6.4f %6.4f %6.4f %6d %6.1f %6.3f %7.2f %7.2f"
          % (r["seq"], r["n_gt_ids"], r["hota"], r["idf1"], r["mota"], r["idsw"],
             r["dwell"]["mape_pct_matched"], r["occupancy"]["mae"],
             r["occupancy"]["mae_pct_of_mean"], r["crossings"]["total_pct_err"]))

# ------------------------------------------- central comparison, across configs
print("\n==== ERROR vs ID-SWITCH RATE (identical detections, association only) ====")
order = sorted(BY_TRACKER, key=lambda c: BY_TRACKER[c]["idsw_per_gt_id_micro"])
X = np.array([BY_TRACKER[c]["idsw_per_gt_id_micro"] for c in order])
SERIES = json.loads(r"""
[["dwell_mape_matched",       "dwell time, per person",      "per-person"],
 ["cross_person_mae_pct",     "crossings, per person",       "per-person"],
 ["occ_mae_pct",              "zone occupancy, per frame",   "aggregate"],
 ["cross_total_pct_err",      "crossings, total",            "aggregate"]]
""")
CORR = {}
print("%-28s %-11s %9s %9s %9s %9s" % ("metric", "kind", "at min", "at max", "pearson", "spearman"))
for key, label, kind in SERIES:
    Y = np.array([BY_TRACKER[c][key] for c in order], dtype=np.float64)
    if len(X) >= 3 and np.std(X) > 0 and np.std(Y) > 0:
        pr = float(pearsonr(X, Y)[0]); sr = float(spearmanr(X, Y)[0])
    else:
        pr = sr = float("nan")
    CORR[key] = {"label": label, "kind": kind, "x": X.tolist(), "y": Y.tolist(),
                 "configs": order, "pearson_r": pr, "spearman_r": sr,
                 "at_lowest_idsw": float(Y[0]), "at_highest_idsw": float(Y[-1]),
                 "ratio_high_over_low": float(Y[-1] / Y[0]) if Y[0] > 1e-9 else float("inf")}
    print("%-28s %-11s %9.2f %9.2f %9.3f %9.3f" % (label, kind, Y[0], Y[-1], pr, sr))
print("")
print("occupancy straight from the detections, NO tracker: MAE %.3f = %.2f%% of mean "
      "occupancy %.2f  (one number per detector, not per tracker config)"
      % (PRIMARY["occ_det_mae"], PRIMARY["occ_det_mae_pct"], PRIMARY["occ_true_mean"]))
CORR["occ_det_mae_pct"] = {
    "label": "zone occupancy, from raw detections (no tracker)",
    "kind": "aggregate-no-tracker",
    "x": X.tolist(), "configs": order,
    "y": [BY_TRACKER[c]["occ_det_mae_pct"] for c in order],
    "pearson_r": None, "spearman_r": None,
    "at_lowest_idsw": float(BY_TRACKER[order[0]]["occ_det_mae_pct"]),
    "at_highest_idsw": float(BY_TRACKER[order[-1]]["occ_det_mae_pct"]),
    "ratio_high_over_low": 1.0}
print("\nID-switches per identity across configs:",
      ", ".join("%s=%.2f" % (c, BY_TRACKER[c]["idsw_per_gt_id_micro"]) for c in order))

# ------------------------------------------- per-identity view (many more points)
print("\n==== PER IDENTITY, pooled over %s sequences, all tracker configs ====" % SCOPE)
ident = [x for r in RUNS if r["split"] == SCOPE and r["det"] == PRIMARY_DET
         for x in r["_rows"] if x["matched"]]
BUCKETS = [(0, 0, "0"), (1, 1, "1"), (2, 2, "2"), (3, 5, "3-5"), (6, 10 ** 9, "6+")]
BUCKET_TABLE = []
print("%-10s %7s %12s %12s %12s %12s"
      % ("ID sw", "n", "dwell MAE s", "dwell MAPE%", "cross MAE", "true dwell s"))
for lo, hi, lab in BUCKETS:
    g = [x for x in ident if lo <= x["idsw"] <= hi]
    if not g:
        continue
    row = {"bucket": lab, "n": len(g),
           "dwell_mae_sec": float(np.mean([x["dwell_abs_err"] for x in g])),
           "dwell_mape": float(np.mean([x["dwell_pct_err"] for x in g])),
           "cross_mae": float(np.mean([x["cross_abs_err"] for x in g])),
           "true_dwell_sec": float(np.mean([x["dwell_true"] for x in g]))}
    BUCKET_TABLE.append(row)
    print("%-10s %7d %12.2f %12.1f %12.3f %12.1f"
          % (lab, row["n"], row["dwell_mae_sec"], row["dwell_mape"],
             row["cross_mae"], row["true_dwell_sec"]))

IDENT_CORR = {}
if len(ident) >= 10:
    xi = np.array([x["idsw"] for x in ident], np.float64)
    for key in ("dwell_pct_err", "dwell_abs_err", "cross_abs_err"):
        yi = np.array([x[key] for x in ident], np.float64)
        if np.std(xi) > 0 and np.std(yi) > 0:
            IDENT_CORR[key] = {"spearman_r": float(spearmanr(xi, yi)[0]),
                               "pearson_r": float(pearsonr(xi, yi)[0]), "n": len(ident)}
            print("  idsw vs %-14s spearman %.3f  pearson %.3f  (n=%d identities)"
                  % (key, IDENT_CORR[key]["spearman_r"], IDENT_CORR[key]["pearson_r"], len(ident)))
    xf = np.array([x["n_pred_ids"] for x in ident], np.float64)
    yf = np.array([x["dwell_pct_err"] for x in ident], np.float64)
    if np.std(xf) > 0 and np.std(yf) > 0:
        IDENT_CORR["n_pred_ids_vs_dwell_pct_err"] = {
            "spearman_r": float(spearmanr(xf, yf)[0]), "n": len(ident)}
        print("  tracks-per-person vs dwell%% err  spearman %.3f"
              % IDENT_CORR["n_pred_ids_vs_dwell_pct_err"]["spearman_r"])

# ---------------------------------------------------------------- the figure
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SURF, INK, INK2 = "#fcfcfb", "#0b0b0b", "#52514e"
HUES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]   # fixed slot order, validated
fig, ax = plt.subplots(1, 2, figsize=(13.5, 5.2), facecolor=SURF,
                       gridspec_kw={"width_ratios": [1.35, 1.0]})
for a in ax:
    a.set_facecolor(SURF)
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        a.spines[s].set_color("#d8d7d2")
    a.tick_params(colors=INK2, labelsize=9, length=3)
    a.grid(axis="y", color="#eceae5", linewidth=0.8)
    a.set_axisbelow(True)

# panel 1 - one y axis; every series is a percentage error of its own true value.
# x is the config order, because the raw rates cluster near zero and would be
# unreadable; the rate each config actually produced is printed under its tick.
xs = np.arange(len(order))
finals = []
for i, (key, label, kind) in enumerate(SERIES):
    Y = [BY_TRACKER[c][key] for c in order]
    ax[0].plot(xs, Y, color=HUES[i], linewidth=2,
               marker="o" if kind == "per-person" else "s",
               markersize=8, markeredgecolor=SURF, markeredgewidth=1.5,
               linestyle="-" if kind == "per-person" else "--", label=label, zorder=3)
    finals.append([Y[-1], label])

# nudge the direct labels apart so they never collide
span = max(max(f[0] for f in finals) - min(f[0] for f in finals), 1.0)
gap = 0.075 * max(span, 40.0)
finals.sort(key=lambda z: z[0])
for i in range(1, len(finals)):
    if finals[i][0] - finals[i - 1][0] < gap:
        finals[i][0] = finals[i - 1][0] + gap
for yy, label in finals:
    ax[0].annotate(label, (xs[-1], yy), xytext=(10, 0), textcoords="offset points",
                   color=INK, fontsize=8.5, va="center")

BASE = PRIMARY["occ_det_mae_pct"]
if np.isfinite(BASE):
    ax[0].axhline(BASE, color="#8a8984", linewidth=1.4, linestyle=(0, (5, 3)), zorder=2)
    ax[0].annotate("zone occupancy from raw detections, no tracker (%.1f%%)" % BASE,
                   (xs[0], BASE), xytext=(0, -6), textcoords="offset points",
                   color=INK2, fontsize=8, va="top", ha="left")

NL = chr(10)
ax[0].set_xticks(xs)
ax[0].set_xticklabels([c + NL + ("%.2f" % BY_TRACKER[c]["idsw_per_gt_id_micro"])
                       for c in order], fontsize=8.5)
ax[0].set_xlim(-0.35, len(order) - 1 + 1.9)
ax[0].set_ylim(bottom=0)
ax[0].set_xlabel("tracker association strength, weakest to the right" + NL +
                 "(ID switches per ground-truth identity under each name)",
                 color=INK2, fontsize=9.5)
ax[0].set_ylabel("error  (% of the true value)", color=INK2, fontsize=10)
ax[0].set_title("Per-person numbers track the ID-switch rate." + NL +
                "The identity-free number does not.",
                color=INK, fontsize=12, loc="left", pad=12)
ax[0].legend(frameon=False, fontsize=8.5, labelcolor=INK2, loc="upper left")

# panel 2 - one measure only, so one axis
labs = [r["bucket"] for r in BUCKET_TABLE]
vals = [r["dwell_mape"] for r in BUCKET_TABLE]
ns = [r["n"] for r in BUCKET_TABLE]
ax[1].bar(range(len(labs)), vals, width=0.68, color=HUES[0], zorder=3)
for i, (v, n) in enumerate(zip(vals, ns)):
    ax[1].annotate("%.0f%%" % v, (i, v), xytext=(0, 4), textcoords="offset points",
                   ha="center", color=INK, fontsize=9.5, fontweight="medium")
    ax[1].annotate("n=%d" % n, (i, 0), xytext=(0, 4), textcoords="offset points",
                   ha="center", color="#ffffff", fontsize=8)
ax[1].set_xticks(range(len(labs)))
ax[1].set_xticklabels(labs)
ax[1].set_xlabel("ID switches suffered by that one person", color=INK2, fontsize=10)
ax[1].set_ylabel("dwell-time error (% of true dwell)", color=INK2, fontsize=10)
ax[1].set_title("Each lost identity costs that person's dwell time",
                color=INK, fontsize=12, loc="left", pad=12)
fig.tight_layout()
FIGPATH = os.path.join(OUT, "idswitch_vs_dashboard_error.png")
fig.savefig(FIGPATH, dpi=160, facecolor=SURF)
print("\nwrote", FIGPATH)
plt.close(fig)


## 12. results.json

In [ ]:
# =====================================================================
# CELL 12 - results.json, with hard assertions so a COMPLETE kernel that
# measured nothing still fails loudly.
# =====================================================================
BARS = CFG["bars"]
bar_idf1 = PRIMARY["idf1"] >= BARS["idf1_min"]
bar_hota = PRIMARY["hota"] >= BARS["hota_min"]
BAR_PASSED = bool(bar_idf1 and bar_hota and HAVE_TEST)

print("=" * 72)
print("BAR CHECK  (%s sequences, %s + %s)" % (SCOPE, PRIMARY_DET, PRIMARY_TRACKER))
print("  IDF1 %.4f  >= %.2f  ->  %s" % (PRIMARY["idf1"], BARS["idf1_min"],
                                        "PASS" if bar_idf1 else "FAIL"))
print("  HOTA %.4f  >= %.2f  ->  %s" % (PRIMARY["hota"], BARS["hota_min"],
                                        "PASS" if bar_hota else "FAIL"))
print("  MOTA %.4f  (leaderboard range for a competent tracker: 0.70-0.80)"
      % PRIMARY["mota"])
print("  held-out test sequences present: %s" % HAVE_TEST)
print("  OVERALL: %s" % ("PASS" if BAR_PASSED else "FAIL / PILOT"))
print("=" * 72)

RESULTS = {
    "project": "cctv-analytics",
    "question": ("which CCTV shop-analytics dashboard numbers are recoverable, "
                 "and which depend on holding a person's identity for the whole visit"),
    "dataset": "MOT17 (ahmedsamir1598/mot17challenge), train split - the only "
               "split that ships ground-truth tracks",
    "split_policy": "by sequence; dev sequences choose the detector, test "
                    "sequences carry every reported number",
    "dev_sequences": CFG["dev_sequences"],
    "test_sequences": CFG["test_sequences"],
    "reporting_scope": SCOPE,
    "pilot_only": bool(PILOT_ONLY),
    "trained_anything": False,
    "detector": {"name": "fasterrcnn_resnet50_fpn_v2",
                 "weights": "COCO_V1 (no MOT17 fine-tuning)",
                 "min_size": CFG["det_min_size"], "max_size": CFG["det_max_size"],
                 "score_keep": CFG["det_score_keep"],
                 "primary_detection_source": PRIMARY_DET,
                 "chosen_on": "dev sequences, by IDF1"},
    "tracker": {"primary": PRIMARY_TRACKER, "configs": TRACKER_CONFIGS,
                "note": "ByteTrack-style IoU + score association with a "
                        "constant-velocity Kalman filter; no appearance model, "
                        "no training"},
    "zone": {"x_frac": CFG["zone_x"], "y_frac": CFG["zone_y"]},
    "counter_line_x_frac": CFG["line_x_frac"],
    "bars": {"idf1_min": BARS["idf1_min"], "hota_min": BARS["hota_min"],
             "idf1": PRIMARY["idf1"], "hota": PRIMARY["hota"],
             "idf1_pass": bool(bar_idf1), "hota_pass": bool(bar_hota),
             "passed": BAR_PASSED},
    "tracking": {k: PRIMARY[k] for k in
                 ("hota", "det_a", "ass_a", "idf1", "mota", "mota_micro", "motp",
                  "recall", "precision", "idsw_total", "idsw_per_gt_id_micro",
                  "idsw_per_1k_gt_dets", "n_gt_dets", "n_gt_ids", "n_sequences",
                  "fn_total", "fp_total")},
    "dashboard": {
        "per_person": {
            "dwell_mae_sec_matched": PRIMARY["dwell_mae_sec_matched"],
            "dwell_mape_matched": PRIMARY["dwell_mape_matched"],
            "dwell_median_pct_matched": PRIMARY["dwell_median_pct_matched"],
            "dwell_mae_sec_all_identities": PRIMARY["dwell_mae_sec_all"],
            "dwell_mape_all_identities": PRIMARY["dwell_mape_all"],
            "dwell_true_mean_sec": PRIMARY["dwell_true_mean_sec"],
            "n_identities": PRIMARY["n_identities"],
            "n_identities_matched": PRIMARY["n_identities_matched"],
            "crossings_mae_per_person": PRIMARY["cross_person_mae"],
            "crossings_mae_per_person_pct": PRIMARY["cross_person_mae_pct"],
            "crossings_true_mean_per_person": PRIMARY["cross_true_mean_per_person"],
            "crossings_exact_fraction": PRIMARY["cross_person_exact_frac"]},
        "aggregate": {
            "zone_occupancy_mae": PRIMARY["occ_mae"],
            "zone_occupancy_mae_pct": PRIMARY["occ_mae_pct"],
            "zone_occupancy_true_mean": PRIMARY["occ_true_mean"],
            "zone_occupancy_pred_mean": PRIMARY["occ_pred_mean"],
            "zone_occupancy_bias": PRIMARY["occ_bias"],
            "zone_occupancy_mae_no_tracker": PRIMARY["occ_det_mae"],
            "zone_occupancy_mae_pct_no_tracker": PRIMARY["occ_det_mae_pct"],
            "zone_occupancy_pred_mean_no_tracker": PRIMARY["occ_det_pred_mean"],
            "crossings_total_true": PRIMARY["cross_total_true"],
            "crossings_total_pred": PRIMARY["cross_total_pred"],
            "crossings_total_pct_err": PRIMARY["cross_total_pct_err"]},
        "naive_dashboard_readout": {
            "visitors_true": PRIMARY["visitors_true"],
            "visitors_reported": PRIMARY["visitors_reported"],
            "visitor_count_pct_err": PRIMARY["visitor_count_pct_err"],
            "mean_dwell_true_sec": PRIMARY["dwell_true_mean_sec"],
            "mean_dwell_reported_sec": PRIMARY["reported_mean_dwell_sec"],
            "mean_dwell_pct_err": PRIMARY["reported_mean_dwell_pct_err"]}},
    "idswitch_relationship": {
        "x_axis": "ID switches per ground-truth identity",
        "tracker_configs_low_to_high_idsw": order,
        "idsw_per_gt_id_by_config": {c: BY_TRACKER[c]["idsw_per_gt_id_micro"]
                                     for c in order},
        "series": CORR,
        "per_identity_buckets": BUCKET_TABLE,
        "per_identity_correlations": IDENT_CORR},
    "per_sequence": PER_SEQ,
    "per_tracker_config": {c: {k: v for k, v in BY_TRACKER[c].items()
                               if not k.startswith("_")} for c in BY_TRACKER},
    "dev_by_detector": DEV_BY_DET,
    "detection_stats": DET_STATS,
    "privacy": {
        "face_recognition": False,
        "enrolment": False,
        "identities": "anonymous integers assigned per sequence, never persisted",
        "note": ("naming a person in such a dashboard requires enrolling that "
                 "person's face; this project does not do that, and the "
                 "per-person productivity claim has no ground truth even when "
                 "the tracking is perfect")},
    "runtime": {"gpu_detection_minutes": GPU_SECONDS / 60.0,
                "wall_minutes": (time.time() - T_START) / 60.0,
                "frames_detected": int(sum(SEQINFO[s]["n_frames"] for s in GT_CACHE)),
                "n_runs": len(RUNS)},
    "figure": os.path.basename(FIGPATH),
}


def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items() if not str(k).startswith("_")}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.bool_, bool)):
        return bool(o)
    if isinstance(o, (np.floating, float)):
        f = float(o)
        return None if (math.isnan(f) or math.isinf(f)) else round(f, 6)
    if isinstance(o, (np.integer, int)):
        return int(o)
    if isinstance(o, (np.bool_, bool)):
        return bool(o)
    return o


RESULTS = _clean(RESULTS)

# ------------------------------------------------- assertions: measured or fail
assert RUNS, "no runs at all"
assert PRIMARY["n_gt_dets"] > 0, "zero ground-truth boxes were evaluated"
assert PRIMARY["n_identities"] > 0, "zero identities were evaluated"
assert PRIMARY["n_sequences"] >= 1, "no sequence was evaluated"
assert PRIMARY["recall"] > 0.0, "the detector matched nothing - detection wiring is broken"
assert np.isfinite(PRIMARY["hota"]) and PRIMARY["hota"] > 0, "HOTA is zero or not finite"
assert np.isfinite(PRIMARY["dwell_mape_matched"]), "dwell error is not finite"
assert PRIMARY["occ_true_mean"] > 0, "the zone is empty in ground truth - bad zone"
assert PRIMARY["cross_total_true"] > 0, "nobody crosses the counter line in GT - bad line"
assert len(BUCKET_TABLE) >= 1, "the per-identity bucket table is empty"
assert os.path.getsize(FIGPATH) > 10000, "the figure did not render"
if not PILOT_ONLY:
    assert len(PER_SEQ) == len(CFG["test_sequences"]), \
        "expected all %d test sequences, got %s" % (len(CFG["test_sequences"]), sorted(PER_SEQ))
    assert len(BY_TRACKER) == len(TRACKER_CONFIGS), "not every tracker config ran"

with open(os.path.join(OUT, "results.json"), "w") as fh:
    json.dump(RESULTS, fh, indent=2, sort_keys=False)
print("\nwrote results.json (%d bytes)"
      % os.path.getsize(os.path.join(OUT, "results.json")))

with open(os.path.join(OUT, "runs_full.json"), "w") as fh:
    json.dump(_clean([{k: v for k, v in r.items() if k != "_rows"} for r in RUNS]), fh, indent=1)

print("\n---- headline ----")
print("HOTA %.4f | IDF1 %.4f | MOTA %.4f | %d ID switches over %d identities"
      % (PRIMARY["hota"], PRIMARY["idf1"], PRIMARY["mota"],
         PRIMARY["idsw_total"], PRIMARY["n_gt_ids"]))
print("dwell time    : MAE %.2f s = %.1f%% of true dwell (true mean %.1f s)"
      % (PRIMARY["dwell_mae_sec_matched"], PRIMARY["dwell_mape_matched"],
         PRIMARY["dwell_true_mean_sec"]))
print("zone occupancy: MAE %.3f people/frame = %.2f%% of mean occupancy %.2f"
      % (PRIMARY["occ_mae"], PRIMARY["occ_mae_pct"], PRIMARY["occ_true_mean"]))
print("crossings     : per person MAE %.3f | total %d vs %d true = %.2f%%"
      % (PRIMARY["cross_person_mae"], PRIMARY["cross_total_pred"],
         PRIMARY["cross_total_true"], PRIMARY["cross_total_pct_err"]))
print("naive readout : %d visitors reported vs %d real (%.1f%%), mean dwell %.1f s vs %.1f s (%.1f%%)"
      % (PRIMARY["visitors_reported"], PRIMARY["visitors_true"],
         PRIMARY["visitor_count_pct_err"], PRIMARY["reported_mean_dwell_sec"],
         PRIMARY["dwell_true_mean_sec"], PRIMARY["reported_mean_dwell_pct_err"]))
print("\nBAR: %s" % ("PASS" if BAR_PASSED else "FAIL / PILOT"))
print("done in %.1f min" % ((time.time() - T_START) / 60.0))
